# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://a09e47e6-93e0-40b6-99ee-5d04e633b955.us-east-1-1.aws.cloud.qdrant.io:6333


## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [2]:
from langchain_core.documents import Document
import fitz

# TODO: PDF 파일 경로를 입력하세요
# 예시: "../datasets/your_document.pdf"
file_path = "../datasets/정보보호 및 개인정보보호 관리체계 인증기준 안내서(2023.11).pdf"

doc = fitz.open(file_path)
docs = []

# 페이지 단위로 Document 생성 (Parent Document)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": file_path.split("/")[-1],
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

총 263개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 215자
평균 페이지 길이: 1819자

첫 페이지 내용 미리보기:
  발 간 등 록 번 호
11-17903650-100016-14


정보보호 및
개인정보보호
관리체계(ISMS-P)
인증기준 안내서





  2023. 11.





             개인정보보호위원회

                                                     personal Information Protection Commission...


## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 263
  - Child chunk 수: 1855
  - 평균 chunk/page: 7.1

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: page_1
  Page: 1
  Length: 213자
  Content: 발 간 등 록 번 호
11-17903650-100016-14


정보보호 및
개인정보보호
관리체계(ISMS-P)
인증기준 안내서





  2023. 11.





      ...

Chunk 2:
  Parent ID: page_2
  Page: 2
  Length: 247자
  Content: 발 간 등 록 번 호
11-17903650-100016-14



   정보보호 및 개인정보보호

 관리체계(ISMS-P) 인증기준 안내서




                  ...

Chunk 3:
  Parent ID: page_3
  Page: 3
  Length: 369자
  Content: 정보보호 및 개인정보보호
 관리체계(ISMS-P) 인증기준 안내서





 안내
 사항




발간 목적

본 안내서는 「정보보호 및 개인정보보호 관리체계 인증 등에 관한 고시」...


## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [4]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://a09e47e6-93e0-40b6-99ee-5d04e633b955.us-east-1-1.aws.cloud.qdrant.io:6333


In [5]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "team1_isms_p"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 'team1_isms_p' 생성 완료

1855개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [6]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 263개의 Parent 문서 저장 완료

Docstore 키 예시: ['page_1', 'page_2', 'page_3', 'page_4', 'page_5']


## 5. Parent Document Retriever 구현

In [8]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

✓ Parent Document Retriever 생성 완료


## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [12]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "개인정보 목적 외 이용 및 제공"

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: 개인정보 목적 외 이용 및 제공


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 224
  Parent ID: page_224
  길이: 328자
  내용: 항 목      3.2.4 개인정보 목적 외 이용 및 제공
                                                                                                  1
         개인정보는 수집 시의 정보주체에게 고지·동의를 받은 목적 또는 법령에 근거한 범위       관
         내에서만 이용 또는 제공하여야 하며, 이를 초과하여 이용·제공하려는 때에는 정보주체의       리체  인증기준         추가 동의를 받거나 관계 법령에 따른 적법한 경우인지 확인하고 적절한 보호대책을       계

Chunk 2:
  페이지: 225
  Parent ID: page_225
  길이: 384자
  내용: ▶개인정보를 목적 외 용도로 이용·제공하기 위하여 동의를 받을 경우 고지사항
     1. 개인정보를 제공받는 자
     2. 개인정보의 이용 목적(제공 시에는 제공받는 자의 이용목적)
     3. 이용 또는 제공하는 개인정보의 항목
     4. 개인정보의 보유 및 이용 기간(제공 시에는 제공받는 자의 보유 및 이용기간)
     5. 동의를 거부할 권리가 있다는 사실 및 동의 거부에 따른 불이익이 있는 경우에 그 불이익의 내용
 ▶다른 개인정보처리자로부터 개인정보를 제공받은 자는 다음 각 호의 어느 하나에 해당하는 경우를
   제외하고는 개인정보를 제공받은 목적 외의 용도로 이용하거나 이를 제3자에게 제공하여서는 아니 됨
    1. 정보주체로부터 별도의 동의를 받은 경우


[2] Parent Document 검색 결과
--------

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [15]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

# TODO: 시스템 프롬프트를 팀 문서 도메인에 맞게 수정하세요
# 예시: "당신은 의료 전문가입니다.", "당신은 법률 전문가입니다." 등
template = """
당신은 ISMS-P 전문가입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.
또한, 답변에 참고한 문서의 출처와 페이지 번호를 명시하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content

print("✓ RAG 시스템 준비 완료")

✓ RAG 시스템 준비 완료


## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [16]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "정보보호 최고책임자(CISO)와 개인정보 보호책임자(CPO)의 지정 요건은 무엇인가요?",
    "조직 규모가 작아 불가피하게 직무 분리가 어려운 경우 어떤 보완 통제 대책을 마련해야 하나요?",
    "사용자가 안전하게 사용할 수 있는 비밀번호 작성 규칙(길이, 조합 등) 기준을 설명해 주세요."
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 정보보호 최고책임자(CISO)와 개인정보 보호책임자(CPO)의 지정 요건은 무엇인가요?



정보보호 최고책임자(CISO)와 개인정보 보호책임자(CPO)는 **최고경영자가 공식적으로 지정**해야 하며, **예산·인력 등 자원을 할당할 수 있는 임원급**이어야 합니다. 또한 각 직무에 필요한 **법적 자격요건**을 충족해야 합니다.

핵심 요건은 다음과 같습니다.

1. **공식 지정**
   - 인사발령 등 **공식 절차**를 통해 지정해야 합니다.
   - 당연직인 경우에도 정보보호/개인정보보호 정책서에 **해당 직위가 명시**되어 있어야 합니다.

2. **임원급 지정**
   - CISO와 CPO는 조직의 정보보호 및 개인정보보호 업무를 실질적으로 총괄할 수 있도록  
     **예산, 인력 등 자원을 할당할 수 있는 임원급**이어야 합니다.

3. **관련 법령상 자격요건 충족**
   - CISO는 **정보통신망법 제45조의3** 및 관련 시행령 요건을 충족해야 합니다.
   - CPO는 **개인정보 보호법 제31조** 및 관련 기준에 따른 지정 요건을 충족해야 합니다.

4. **업무 총괄 가능성**
   - CISO는 정보보호 계획 수립·시행, 정기 감사 및 개선, 위험 식별·평가 및 대책 마련 등을 총괄해야 합니다.
   - CPO는 개인정보 보호 계획 수립·시행, 처리실태 점검, 불만 처리 및 피해구제, 유출 방지 내부통제 등을 총괄해야 합니다.

### 참고한 출처
- **정보보호 및 개인정보보호 관리체계 인증기준 안내서(2023.11)**, p.17  
  - 최고책임자 지정 요건, 임원급 지정, 공식 지정 및 자격요건 설명
- **정보보호 및 개인정보보호 관리체계 인증기준 안내서(2023.11)**, p.57  
  - CISO/CPO의 역할과 책임, 조직 운영상 총괄·관리 요건 설명

원하시면 제가 이 내용을 **ISMS-P 인증 심사 관점의 답변 형태**로 더 간단하게 정리해드릴게요.


질문: 조직 규모가 작아 불가피하게 직무 분리가 어려운 경우 어떤 보완 통제 대책을 마련해야 하나요?



조직 규모가 작아 **직무 분리가 어려운 경우**, ISMS-P에서는 직무 오·남용을 예방하기 위한 **보완 통제**를 마련하도록 요구합니다.  
주요 보완 통제는 아래와 같습니다.

1. **직무자 간 상호 검토**
   - 한 사람이 단독으로 처리하지 않도록 다른 직무자가 교차로 검토합니다.
   - 예: 개발 반영, 권한 변경, 운영 설정 변경 등을 다른 담당자가 확인

2. **상위관리자 정기 모니터링 및 승인**
   - 변경사항이나 주요 작업은 상위관리자가 정기적으로 점검하고 승인합니다.
   - 예: 계정 생성/삭제, 권한 부여, 시스템 설정 변경, 배포 작업 승인

3. **책임추적성 확보**
   - 누가, 언제, 무엇을 했는지 추적 가능하도록 기록을 남깁니다.
   - 예: 개인별 계정 사용, 로그기록, 감사·모니터링 체계 운영

4. **개인별 계정 사용 원칙 적용**
   - 공용계정 대신 개인 계정을 사용하여 행위 주체를 명확히 합니다.

5. **변경사항 승인 절차 마련**
   - 직무 분리가 어려운 경우에도 주요 변경은 사전 승인 후 수행하도록 합니다.

즉, 핵심은 **“혼자 처리하지 않게 하고, 반드시 검토·승인·기록이 남도록 하는 것”**입니다.  
이러한 보완 통제를 통해 직무 분리 미흡으로 인한 권한 오·남용 위험을 줄여야 합니다.

**참고 문서 및 페이지**
- **정보보호 및 개인정보보호 관리체계 인증기준 안내서(2023.11).pdf, p.62**
  - 항목 2.2.2 직무 분리
  - “직무자 간 상호 검토, 상위관리자 정기 모니터링 및 변경사항 승인, 책임추적성 확보 방안” 관련 설명




질문: 사용자가 안전하게 사용할 수 있는 비밀번호 작성 규칙(길이, 조합 등) 기준을 설명해 주세요.



사용자가 안전하게 사용할 수 있는 비밀번호 작성 규칙은, **서비스의 특성 및 위험도**를 고려하되, 일반적으로 다음 기준을 적용하도록 안내하고 있습니다.

### 1) 비밀번호 길이 및 조합 기준
- **영문, 숫자, 특수문자 중 2종류 이상을 조합하여 최소 8자리 이상**
- **문자로만 구성하는 경우 최소 10자리 이상**
  - 단, **숫자만으로 구성한 비밀번호는 취약할 수 있으므로 지양**해야 합니다.

### 2) 추측하기 쉬운 비밀번호 제한
다음과 같이 **예측 가능한 비밀번호는 사용을 제한**해야 합니다.
- 동일한 문자 반복
- 키보드 상에서 나란히 있는 문자열
- 일련번호
- 연속적인 숫자
- 생일, 전화번호 등 쉽게 추측 가능한 개인정보
- ID와 유사한 비밀번호

### 3) 비밀번호 변경 및 재사용 제한
- **비밀번호 유효기간을 설정하여 주기적으로 변경**하도록 할 수 있습니다.  
  다만, **변경 여부 및 변경 주기는 위험평가 결과 등을 고려하여 자체적으로 결정**합니다.
- **비밀번호 변경 시 이전에 사용한 비밀번호의 재사용을 제한**해야 합니다.

### 4) 적용 시 유의사항
- 위 기준은 **불가피한 경우를 제외하고 시스템적으로 강제화**하는 것이 바람직합니다.
- 사용자뿐 아니라 **개인정보취급자 비밀번호**에도 동일한 수준의 관리가 필요합니다.

---

### 참고한 출처
- **정보보호 및 개인정보보호 관리체계 인증기준 안내서(2023.11).pdf**
  - **페이지 99**
  - **페이지 100**



## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합